# EGC 21818 - Projeto final: PPG do pulso x ECG no WESAD
Notebook de **reprodução**: refaz a rodada dos 15 participantes e a análise. Rode a célula 0 a cada sessão; depois as demais em ordem. Se o Colab desconectar durante a rodada, rode a célula 0 e a rodada de novo: quem já terminou é pulado.

O passo a passo exploratório (S10 e S11) está em `notebooks/celula_passo*.py` no repositório. Explicação do método e dos resultados: `docs/`.

## 0. Preparação

In [ ]:
import sys, os
from google.colab import drive, files
drive.mount('/content/drive')

RAIZ_REPO   = '/content/drive/MyDrive/projeto_egc'
ZIP_WESAD   = '/content/drive/MyDrive/WESAD.zip'
PASTA_DADOS = '/content/dados/WESAD'      # temporária: o código extrai e apaga cada .pkl

# Código: fica no Drive. Só pede o zip do projeto se ainda não estiver lá.
if not os.path.exists(RAIZ_REPO + '/src/rodada_final.py'):
    print('Código não encontrado no Drive. Escolha o zip do projeto (ex.: projeto_egc_v9.zip):')
    enviado = files.upload()
    nome_zip = list(enviado)[0]
    !unzip -o -q "{nome_zip}" -d /content/drive/MyDrive/

sys.path.insert(0, RAIZ_REPO + '/src')
import pywt
print('PyWavelets', pywt.__version__)
assert os.path.exists(ZIP_WESAD), 'WESAD.zip não encontrado no Drive'
assert os.path.exists(RAIZ_REPO + '/src/rodada_final.py'), 'código não encontrado'
print('Tudo pronto.')

## 1. Testes (opcional, recomendado)
Provam que o código faz o que o texto diz em sinais conhecidos; não provam que funciona nos dados reais.

In [ ]:
# Testes do código (sinal sintético ou arquivos fictícios)
!cd {RAIZ_REPO} && for t in testes/teste_*.py; do echo "== $t"; python -I $t 2>&1 | tail -1; done

## 2. Rodada dos 15 participantes
Cerca de 2 a 3 min por participante. Um CSV por participante em `resultados/final/` (checkpoint).

In [ ]:
# Passo 18: rodar os 15 participantes (um de cada vez, com checkpoint no Drive)
from rodada_final import rodar_participantes, TODOS

PASTA_RES = RAIZ_REPO + '/resultados/final'
estado = rodar_participantes(TODOS, ZIP_WESAD, PASTA_DADOS, PASTA_RES)

## 3. Tabelas e figuras finais

In [ ]:
# Passo 19: tabelas e figuras finais (lê os CSV de resultados/final)
from rodada_final import TODOS
from analise import gerar_tudo

PASTA_RES = RAIZ_REPO + '/resultados/final'
t = gerar_tudo(PASTA_RES, esperados=TODOS)

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob(RAIZ_REPO + '/resultados/final/figuras/*.png')):
    print(f.split('/')[-1])
    display(Image(f))

## 4. Checagens usadas no relatório
Resultado sem S10 e S11 (participantes de desenvolvimento) e coerência do RMSSD.

In [ ]:
# Checagens usadas no relatório (lê os CSV de resultados/final)
import pandas as pd, glob
df = pd.concat([pd.read_csv(f) for f in glob.glob(RAIZ_REPO + '/resultados/final/janelas_*.csv')])

# 1) sem os participantes de desenvolvimento: média por participante, depois entre participantes
sem = df[~df.sujeito.isin(['S10', 'S11'])]
pp = sem.groupby(['sujeito', 'condicao', 'braco'])[['se', 'pp', 'se_acaso']].mean()
print(pp.groupby(['condicao', 'braco']).mean().round(3), '\n')

# 2) RMSSD: valores absolutos (passa-banda) e erro por faixa de Se
r = df[df.braco == 'passa-banda'].copy()
print(r.groupby('condicao')[['rmssd_ecg', 'rmssd_ppg', 'n_ecg', 'n_ppg']].describe().T.round(1))
r['erro'] = (r.rmssd_ppg - r.rmssd_ecg).abs()
r['faixa_se'] = pd.cut(r.se, [0, .6, .7, .8, .9, 1.0])
print(r.groupby('faixa_se', observed=False)[['erro', 'rmssd_ecg', 'rmssd_ppg']].agg(['median', 'count']).round(0))